### Block 1 — Setup & Config

In [59]:
# !pip install -U pandas fsspec s3fs pyarrow

import os, re
import pandas as pd
import fsspec

# === CONFIG ===
WEATHER = "TMY3"        # or "AMY2018"
CHUNK_ROWS = 200_000

OUT_CSV     = "resstock2024_2_national_baseline_filtered_FULL_COLUMNS.csv"
OUT_PARQUET = "resstock2024_2_national_baseline_filtered_FULL_COLUMNS.parquet"
SAMPLE_N    = 100
SAMPLE_CSV  = "sample_preview.csv"


### Block 2 — Row Filters (rules)

In [61]:
# If you want to load rules from a file (Field|Rule|Value), set the path here:
RULES_FILE = None

if RULES_FILE:
    rules = pd.read_excel(RULES_FILE) if RULES_FILE.lower().endswith((".xlsx",".xls")) else pd.read_csv(RULES_FILE)
else:
    rules = pd.DataFrame({
        "Field": [
            "in.ashrae_iecc_climate_zone_2004",
            "in.hvac_cooling_type",
            "in.ceiling_fan",
            "in.geometry_building_type_recs",
            "in.vacancy_status",
            "in.geometry_wall_type",
        ],
        "Rule":  ["incl","incl","incl","excl","incl","excl"],
        "Value": [
            "5A",
            "Room AC or None, Non-Ducted, Heat Pump",
            "None or Standard Efficiency, No usage",
            "Mobile Home",
            "Occupied",
            "Wood Frame",
        ],
    })

rules = rules[["Field","Rule","Value"]].copy()
rules["Field"] = rules["Field"].astype(str).str.strip()
rules["Rule"]  = rules["Rule"].astype(str).str.strip().str.lower()
rules["Value"] = rules["Value"].astype(str).str.strip()
assert set(rules["Rule"]).issubset({"incl","excl"})
rules


,Field,Rule,Value
0,in.ashrae_iecc_climate_zone_2004,incl,5A
1,in.hvac_cooling_type,incl,"Room AC or None, Non-Ducted, Heat Pump"
2,in.ceiling_fan,incl,"None or Standard Efficiency, No usage"
3,in.geometry_building_type_recs,excl,Mobile Home
4,in.vacancy_status,incl,Occupied
5,in.geometry_wall_type,excl,Wood Frame


### Block 3 — National Baseline CSV (2024.2)

In [63]:
BASE = "s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/"
RELEASE = "resstock_amy2018_release_2" if WEATHER.upper()=="AMY2018" else "resstock_tmy3_release_2"

NATIONAL_BASELINE = (
    BASE + f"{RELEASE}/metadata_and_annual_results/national/csv/"
    "baseline_metadata_and_annual_results.csv"
)

fs = fsspec.filesystem("s3", anon=True)
assert fs.exists(NATIONAL_BASELINE), f"File not found: {NATIONAL_BASELINE}"
print("USING NATIONAL BASELINE:", NATIONAL_BASELINE)

# Single-file list for a unified downstream flow
files = [NATIONAL_BASELINE]

# Sanity check: header is readable
_ = pd.read_csv(files[0], nrows=0, storage_options={"anon": True})
print("OK — national baseline is available.")


USING NATIONAL BASELINE: s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/metadata_and_annual_results/national/csv/baseline_metadata_and_annual_results.csv
OK — national baseline is available.


### Block 4 — Build Row Filter (incl/excl) with HVAC normalization

In [65]:
NULL_TOKENS = {"", "none", "null", "na", "nan"}

def normalize_token(field: str, s: str) -> str:
    x = str(s).strip().lower()
    if field == "in.hvac_cooling_type":
        # collapse spaces/commas and normalize “Non-Ducted Heat Pump”
        x = re.sub(r"[\s,]+", " ", x)
        if re.search(r"non-?ducted\s+heat\s*pump", x):
            x = "non-ducted heat pump"
    return x

def parse_values_for_field(field: str, v: str):
    s = re.sub(r"\s+or\s+", "|", str(v), flags=re.I)
    if field == "in.hvac_cooling_type":
        s = s.replace(",", "|")
    parts = [p.strip() for p in s.split("|") if p.strip()]
    vals  = [normalize_token(field, p) for p in parts]
    if field == "in.hvac_cooling_type" and "non-ducted" in vals and "heat pump" in vals:
        vals.append("non-ducted heat pump")
    return list(dict.fromkeys(vals))

def build_filter(rules_df: pd.DataFrame):
    incl = rules_df[rules_df["Rule"]=="incl"].copy()
    excl = rules_df[rules_df["Rule"]=="excl"].copy()

    incl_rules = [(f, parse_values_for_field(f, v)) for f, v in zip(incl["Field"], incl["Value"])]
    excl_rules = [(f, parse_values_for_field(f, v)) for f, v in zip(excl["Field"], excl["Value"])] if len(excl) else []

    def row_mask(df: pd.DataFrame) -> pd.Series:
        mask = pd.Series(True, index=df.index)

        # INCL: all conditions must match
        for col, vals in incl_rules:
            if col not in df.columns:
                return pd.Series(False, index=df.index)
            s_norm = df[col].astype(str).fillna("").apply(lambda x: normalize_token(col, x))
            mcol = pd.Series(False, index=df.index)
            for v in vals:
                if v in NULL_TOKENS:
                    mcol |= s_norm.isin(["", "none", "null", "na", "nan"])
                else:
                    mcol |= (s_norm == v)
            mask &= mcol

        # EXCL: none of the values may match
        for col, vals in excl_rules:
            if col not in df.columns:
                continue
            s_norm = df[col].astype(str).fillna("").apply(lambda x: normalize_token(col, x))
            mex = pd.Series(False, index=df.index)
            for v in vals:
                if v in NULL_TOKENS:
                    mex |= s_norm.isin(["", "none", "null", "na", "nan"])
                else:
                    mex |= (s_norm == v)
            mask &= (~mex)

        return mask

    return row_mask

row_filter = build_filter(rules)
print("Filter built. INCL:", sum(rules["Rule"].eq("incl")), "| EXCL:", sum(rules["Rule"].eq("excl")))


Filter built. INCL: 4 | EXCL: 2


### Block 5 — Full Schema & Annual Electricity Column Detection

In [68]:
# Full list of columns from the national baseline
ALL_COLS = list(pd.read_csv(files[0], nrows=0, storage_options={"anon": True}).columns)
print(f"Columns in file: {len(ALL_COLS)}")

# Priority patterns for annual electricity (kWh)
ENERGY_PRIORITY_PATTERNS = [
    r"^out\.total\.electricity\.annual_kwh$",
    r"^out\.electricity\.total\.annual_kwh$",
    r"^site\.electricity\.annual_kwh$",
    r"^out\.electricity\.annual_kwh$",
    r"^out\.electricity\.total\.energy_consumption$",  # lowest priority fallback
]

import re
def pick_energy_column(cols):
    for pat in ENERGY_PRIORITY_PATTERNS:
        for c in cols:
            if re.search(pat, c, flags=re.I):
                return c
    return None

ENERGY_COL = pick_energy_column(ALL_COLS)
print("Detected annual electricity column:", ENERGY_COL)

EXTRA_COL = "annual_electricity_kwh_resolved"
ALL_COLS_PLUS = ALL_COLS + ([EXTRA_COL] if EXTRA_COL not in ALL_COLS else [])


Columns in file: 289
Detected annual electricity column: None


### Block 6 — Read All Columns → Apply Row Filters → Resolve Annual kWh → Write

In [71]:
# Clean previous outputs
for p in (OUT_CSV, OUT_PARQUET, SAMPLE_CSV):
    if os.path.exists(p): os.remove(p)

total_in = total_out = 0
header_written = False
sample_saved = False
sample_buffer = []

reader = pd.read_csv(
    files[0],
    dtype=str,
    low_memory=False,
    chunksize=CHUNK_ROWS,
    storage_options={"anon": True},
    # read all columns (no usecols)
)

for ch in reader:
    total_in += len(ch)

    # 1) apply row filters only
    m = row_filter(ch)
    sub = ch.loc[m, :].copy()

    # 2) resolve annual electricity column and create unified numeric field
    if ENERGY_COL is not None and ENERGY_COL in sub.columns:
        sub[EXTRA_COL] = pd.to_numeric(sub[ENERGY_COL], errors="coerce")
    else:
        sub[EXTRA_COL] = pd.NA

    # 3) align to full schema + EXTRA_COL
    for c in ALL_COLS_PLUS:
        if c not in sub.columns:
            sub[c] = pd.NA
    sub = sub[ALL_COLS_PLUS]

    total_out += len(sub)

    if len(sub):
        sub.to_csv(OUT_CSV, mode="a", index=False, header=not header_written)
        header_written = True

        if not sample_saved:
            sample_buffer.append(sub.head(SAMPLE_N - sum(len(x) for x in sample_buffer)))
            if sum(len(x) for x in sample_buffer) >= SAMPLE_N:
                pd.concat(sample_buffer, ignore_index=True).head(SAMPLE_N).to_csv(SAMPLE_CSV, index=False)
                sample_saved = True

print(f"Done. Input rows: {total_in:,} | After filter: {total_out:,}")
print("CSV ->", OUT_CSV)
if sample_saved:
    print(f"Sample -> {SAMPLE_CSV} (first {SAMPLE_N} rows)")


Done. Input rows: 549,715 | After filter: 4,435
CSV -> resstock2024_2_national_baseline_filtered_FULL_COLUMNS.csv
Sample -> sample_preview.csv (first 100 rows)


### Block 7 — Optional: Parquet Export

In [73]:
try:
    df_all = pd.read_csv(OUT_CSV, dtype=str, low_memory=False)
    df_all.to_parquet(OUT_PARQUET, index=False)
    print("Parquet ->", OUT_PARQUET, "| Rows:", len(df_all))
except Exception as e:
    print("Parquet skip:", e)


Parquet -> resstock2024_2_national_baseline_filtered_FULL_COLUMNS.parquet | Rows: 4435


### Block 8 — Column Filter & Export (9 columns)

In [75]:
# This block reads the previously created OUT_CSV, keeps exactly the 9 FINAL_COLS
# (in this order), ensures the energy column is numeric, and writes a new CSV.

import os
import pandas as pd
import re
import fsspec

# ---- Inputs / Outputs ----
try:
    IN_CSV = OUT_CSV  # reuse from previous blocks if defined
except NameError:
    IN_CSV = "resstock2024_2_national_baseline_filtered_FULL_COLUMNS.csv"

OUT_CSV_FINAL = "resstock2024_2_national_baseline_FINAL_9cols.csv"
CHUNK_ROWS_8 = 200_000

# ---- Required final column list (exact order) ----
FINAL_COLS = [
    "bldg_id",
    "in.bedrooms",
    "in.clothes_washer_presence",
    "in.dishwasher",
    "in.geometry_building_type_recs",
    "in.heating_fuel",
    "in.hvac_cooling_type",
    "in.misc_freezer",
    "out.electricity.total.energy_consumption.kwh",
]

# ---- Helper: find an annual electricity column if the standard name is absent ----
ENERGY_CANDIDATE_PATTERNS = [
    r"^out\.electricity\.total\.energy_consumption\.kwh$",
    r"^out\.total\.electricity\.annual_kwh$",
    r"^out\.electricity\.total\.annual_kwh$",
    r"^site\.electricity\.annual_kwh$",
    r"^out\.electricity\.annual_kwh$",
    r"^out\.electricity\.total\.energy_consumption$",
    r"^annual_electricity_kwh_resolved$",
]

def detect_energy_col(columns):
    cols = list(columns)
    for pat in ENERGY_CANDIDATE_PATTERNS:
        for c in cols:
            if re.search(pat, c, flags=re.I):
                return c
    return None

# ---- Stream, filter, rename, and write ----
if os.path.exists(OUT_CSV_FINAL):
    os.remove(OUT_CSV_FINAL)

total_rows_in = total_rows_out = 0
header_written = False

# First, peek header to figure out which energy column we have
hdr_cols = list(pd.read_csv(IN_CSV, nrows=0).columns)
energy_src_col = (
    "out.electricity.total.energy_consumption.kwh"
    if "out.electricity.total.energy_consumption.kwh" in hdr_cols
    else detect_energy_col(hdr_cols)
)

if energy_src_col is None:
    raise RuntimeError(
        "Could not find an annual electricity column in the input file. "
        "Expected one of: " + ", ".join(ENERGY_CANDIDATE_PATTERNS)
    )

# We will read only the needed columns (plus the detected energy source column if different)
read_cols = set(FINAL_COLS)
read_cols.add(energy_src_col)

reader = pd.read_csv(IN_CSV, chunksize=CHUNK_ROWS_8, usecols=lambda c: c in read_cols, low_memory=False)

for chunk in reader:
    total_rows_in += len(chunk)

    # If the energy column name differs from the target, create/rename it
    if energy_src_col != "out.electricity.total.energy_consumption.kwh":
        chunk["out.electricity.total.energy_consumption.kwh"] = pd.to_numeric(
            chunk[energy_src_col], errors="coerce"
        )
    else:
        chunk["out.electricity.total.energy_consumption.kwh"] = pd.to_numeric(
            chunk["out.electricity.total.energy_consumption.kwh"], errors="coerce"
        )

    # Ensure all FINAL_COLS exist (add missing as NA), then enforce order
    for col in FINAL_COLS:
        if col not in chunk.columns:
            chunk[col] = pd.NA
    chunk = chunk[FINAL_COLS]

    total_rows_out += len(chunk)
    chunk.to_csv(OUT_CSV_FINAL, mode="a", index=False, header=not header_written)
    header_written = True

print(f"[Block 8] Done. Input rows: {total_rows_in:,} | Output rows: {total_rows_out:,}")
print("Filtered CSV ->", OUT_CSV_FINAL)


[Block 8] Done. Input rows: 4,435 | Output rows: 4,435
Filtered CSV -> resstock2024_2_national_baseline_FINAL_9cols.csv
